Need for RNN (Recurrent Neural Network)

Sequential data -> e.g -> movie reviews (can be of variable length and every scentences word_order has meaning)

The ANN & CNN can't work properly on sequential data b/c of two main reasons -

    1. Sequence can be of any length ... whereas ANN are are fixed length (whereas zero padding apporach can be used -> but it increases the cost of computation)
    2. Sequence contains some meaning ... whereas ANN every word is send together due to which we losse the meaning of sequence.

RNN are special types of NN which has a memory ka feature -> they can keep track of past inputs...

![lec_img/rnn.png](lec_img/rnn.png)

In [2]:
from keras import Sequential
from keras.layers import Dense, SimpleRNN

In [3]:
model = Sequential()

model.add(SimpleRNN(3, input_shape=(4,5))) #   #nodes=3, #timestamp=4, #inputs=5
model.add(Dense(1, activation='sigmoid'))

model.summary()

2025-09-04 15:34:20.404845: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2025-09-04 15:34:20.404871: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-09-04 15:34:20.404873: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-09-04 15:34:20.404902: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-04 15:34:20.404909: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/Users/praveenk/Documents/Deep_learning/tf-metal/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. 

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 3)              │            27 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)

How the RNN works behind the scenes ->

Lets say we have movie review as following -> now every unique word throughout the review section is collected and one hot encoding is applied on it -> so that we can represent it using the numeric form

![lec_img/rnn_working_model.png](lec_img/rnn_working_model.png)

Now the input to the model is given on timestamp basis -> 

so lets say we have to input the first review -> {movie was good}= [10000, 01000, 00100]
- At [t=1] -> we have   [1 0 0 0 0]    as input across 3 nodes -> and this nodes has inital bias as (0 or random)

And to retain the meaning of the scentence ... the next input 
- At [t=2] -> we have   [0 1 0 0 0]    as input and we take the bias from [t=1] and create new bias for nodes at [t=2] 

Similarly for the next word
- At [t=3] -> we have   [0 0 1 0 0]    as input and we take the bias from [t=2] and create new bias for nodes at [t=3]

after collecting these bias at the 3 nodes they are then used to compute the final output node

The weights and bias during these are like - 

During forming of bias -> it is combination of (3*3) for each 3 nodes at every timestamp. -> represented by the RGB above

Now lets look at each of the trainable parameters....

In [9]:
# Trainable parameters b/w input and the node
print(model.get_weights()[0].shape)
model.get_weights()[0]

(5, 3)


array([[-0.06978822,  0.60365003, -0.07374722],
       [-0.12500173,  0.6326111 ,  0.70133954],
       [-0.45838246, -0.42715907, -0.44883746],
       [-0.07727486, -0.28544372,  0.5328123 ],
       [ 0.27913648,  0.40314728,  0.8125593 ]], dtype=float32)

In [15]:
# weights at feedback connection RGB
print(model.get_weights()[1].shape)
model.get_weights()[1]

(3, 3)


array([[ 0.39157498, -0.2772154 ,  0.87739426],
       [-0.5105554 ,  0.7278299 ,  0.45781738],
       [ 0.7655078 ,  0.62722814, -0.14346623]], dtype=float32)

![lec_img/output_rnn.png](lec_img/output_rnn.png)

In [16]:
# for the bias at the hidden nodes
print(model.get_weights()[2].shape)
model.get_weights()[2]

(3,)


array([0., 0., 0.], dtype=float32)

In [17]:
# weights for the hidden nodes and the output node
print(model.get_weights()[3].shape)
model.get_weights()[3]

(3, 1)


array([[-0.86860853],
       [ 0.02220118],
       [ 0.5734378 ]], dtype=float32)

In [18]:
# bias at the output node
print(model.get_weights()[4].shape)
model.get_weights()[4]

(1,)


array([0.], dtype=float32)

## Why the word recurrent in RNN -> 
b/c we are again and again using the same hidden node for each timestamp to calculate the output

Also you are giving different inputs at each timestamp but the same weigths are used ->i.e. weight sharing concept is used.

![lec_img/simplified_rnn.png](lec_img/simplified_rnn.png)

## Working -> 

The hidden node has the activation function ... which takes input x(i,t) and output from the prev timestamp with weight metrics as(Wi & Wh) ... then the dot product of both is added and this vector is then send to the function -> throught which we recieve output for the current timestamp.

And for the last timestamp the Ot is given to the weight metrics(Wo) ... and the dot product of this is given to the output function ...

This output function can vary depending on the problem ->

    1. Binary classification -> Sigmoid
    2. Multiclass classifcaiton -> Softmax
    3. Regression -> linear
